In [16]:
import pandas as pd
import numpy as np

orders_full = pd.read_csv('../data_processed/orders_full_clean.csv')
orders_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  str    
 1   customer_id                    113425 non-null  str    
 2   order_status                   113425 non-null  str    
 3   order_purchase_timestamp       113425 non-null  str    
 4   order_approved_at              113264 non-null  str    
 5   order_delivered_carrier_date   111457 non-null  str    
 6   order_delivered_customer_date  110196 non-null  str    
 7   order_estimated_delivery_date  113425 non-null  str    
 8   order_item_id                  112650 non-null  float64
 9   product_id                     112650 non-null  str    
 10  seller_id                      112650 non-null  str    
 11  shipping_limit_date            112650 non-null  str    
 12  price                          112650 non

In [17]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
             'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_cols:
    orders_full[col] = pd.to_datetime(orders_full[col])

orders_full[date_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 5 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_purchase_timestamp       113425 non-null  datetime64[us]
 1   order_approved_at              113264 non-null  datetime64[us]
 2   order_delivered_carrier_date   111457 non-null  datetime64[us]
 3   order_delivered_customer_date  110196 non-null  datetime64[us]
 4   order_estimated_delivery_date  113425 non-null  datetime64[us]
dtypes: datetime64[us](5)
memory usage: 4.3 MB


In [18]:
orders_full['delivery_delay_days'] = (orders_full['order_delivered_customer_date'] - orders_full['order_estimated_delivery_date']).dt.days

orders_full['on_time'] = orders_full['delivery_delay_days'] <= 0

orders_full['on_time'].value_counts()
orders_full['on_time'].mean()

np.float64(0.9074807141282786)

In [19]:
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

# Merge review score (keep one review per order — average if somehow duplicated)
review_scores = order_reviews.groupby('order_id')['review_score'].mean().reset_index()
orders_full = orders_full.merge(review_scores, on='order_id', how='left')

# Merge customer state (for regional analysis)
orders_full = orders_full.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')

orders_full[['review_score', 'customer_state']].info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   review_score    112464 non-null  float64
 1   customer_state  113425 non-null  str    
dtypes: float64(1), str(1)
memory usage: 2.0 MB


In [20]:
region_map = {
    'AC': 'North', 'AP': 'North', 'AM': 'North', 'PA': 'North', 'RO': 'North', 'RR': 'North', 'TO': 'North',
    'AL': 'Northeast', 'BA': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast',
    'PE': 'Northeast', 'PI': 'Northeast', 'RN': 'Northeast', 'SE': 'Northeast',
    'DF': 'Central-West', 'GO': 'Central-West', 'MT': 'Central-West', 'MS': 'Central-West',
    'ES': 'Southeast', 'MG': 'Southeast', 'RJ': 'Southeast', 'SP': 'Southeast',
    'PR': 'South', 'RS': 'South', 'SC': 'South'
}

orders_full['region'] = orders_full['customer_state'].map(region_map)

# On-time rate and avg review score by region
region_stats = orders_full.groupby('region').agg(
    on_time_rate=('on_time', 'mean'),
    avg_review_score=('review_score', 'mean'),
    order_count=('order_id', 'count')
).sort_values('on_time_rate', ascending=False)

region_stats

,on_time_rate,avg_review_score,order_count
region,,,
South,0.920711,4.045643,16257
Southeast,0.913637,4.040321,77973
Central-West,0.911292,3.989897,6651
North,0.886856,3.905945,2077
Northeast,0.842744,3.843750,10467


In [ ]:
# Only look at delivered orders with a valid delay value
delivered = orders_full.dropna(subset=['delivery_delay_days', 'review_score'])

# Bucket delay into readable groups
def delay_bucket(days):
    if days <= -7:
        return 'Very early'
    elif days <= 0:
        return 'On time'
    elif days <= 7:
        return 'Slightly late'
    else:
        return 'Very late'

delivered['delay_bucket'] = delivered['delivery_delay_days'].apply(delay_bucket)

review_by_delay = delivered.groupby('delay_bucket')['review_score'].mean().reindex(
    ['Very early', 'On time', 'Slightly late', 'Very late']
)

review_by_delay#

delay_bucket
Very early       4.225786
On time          4.109591
Slightly late    2.680848
Very late        1.698330
Name: review_score, dtype: float64